# Lesson 2: The Attached AGA Workflow End to End

**Module:** AGA Fundamentals  
**Dataset:** Movies (Actors, Movies, Genres, Users)

This notebook runs the full attached AGA workflow: estimate session memory, create a session bound to your AuraDB instance, project a graph, exercise the three execution modes (`stream`, `mutate`, `write`), verify the writeback, and clean up.

By the end you'll have produced a script you could schedule against any AuraDB.

## Step 1 — Setup

Same env-loading pattern as `1_connect.ipynb`, with the addition of `AlgorithmCategory` (used by the memory estimator in the next step).

In [ ]:
# Imports and environment variables
import os
import pandas as pd
from datetime import timedelta
from IPython.display import display
from graphdatascience.session import GdsSessions, AuraAPICredentials
from graphdatascience.session import DbmsConnectionInfo, AlgorithmCategory, SessionMemory
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Get Aura API credentials
client_id = os.getenv('AURA_CLIENT_ID')
client_secret = os.getenv('AURA_CLIENT_SECRET')

# Get AuraDB connection info
uri = os.getenv('AURA_URI')
username = os.getenv('AURA_USERNAME')
password = os.getenv('AURA_PASSWORD')

In [ ]:
# Create sessions manager
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(
        client_id,
        client_secret
    )
)

## Step 2 — Estimate the session memory

`sessions.estimate(...)` takes rough node and relationship counts plus the algorithm categories you plan to run, and returns the right memory tier. Skip this and you'll either get a session too small for the projection or one larger than you need.

We're projecting `Actor-COLLABORATED-Actor` and running a centrality algorithm.

In [ ]:
memory = sessions.estimate(
    node_count=20_000,
    relationship_count=100_000,
    algorithm_categories=[AlgorithmCategory.CENTRALITY],
)
print(f"Estimator recommends: {memory}")

For a graph of this size the estimator will typically return `SessionMemory.m_2GB` or `m_4GB`. We'll pass that value straight into `get_or_create` next.

## Step 3 — Create the session

Attach to your AuraDB instance using the memory tier the estimator returned. 30-minute TTL is plenty for this notebook.

In [ ]:
# Create a GDS Session
gds = sessions.get_or_create(
    session_name="attached-demo",
    memory=memory,
    db_connection=DbmsConnectionInfo(
        uri=uri,
        username=username,
        password=password
    ),
    ttl=timedelta(minutes=30)
)
print(f"Client ID starts with: {client_id[:12]}...")
gds.verify_connectivity()
print(f"Connected to GDS Session: attached-demo")

The session is now running. The `gds` object is your interface to it — the same `GraphDataScience` class you've used against a local Neo4j or AuraDS instance.

## Step 4 — Run a Cypher query

`gds.run_cypher(...)` sends Cypher to the AuraDB the session is attached to and returns the result as a pandas DataFrame.

In [ ]:
# Quick check on the database contents
df = gds.run_cypher("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(*) AS count
    ORDER BY count DESC
""")
display(df)

Each row is a node label and its count in the connected database. This is your sanity check that the session reached AuraDB and ran the query.

## Step 5 — Project a graph

Project an `Actor-COLLABORATED-Actor` graph: two actors are connected if they've appeared in at least one movie together, weighted by how many movies they share.

The Cypher inside `gds.graph.project()` describes which subgraph to project; the `gds.graph.project.remote(...)` call inside the query is the AGA-specific procedure that lands the data in the session over Arrow Flight.

In [ ]:
# Project actor collaborations using remote projection
G, result = gds.graph.project(
    "actor-collaborations",
    """
    CALL {
      MATCH (a1:Actor)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor)
      WHERE a1 <> a2
      RETURN a1 AS source,
             a2 AS target,
             a1{} AS sourceNodeProperties,
             a2{} AS targetNodeProperties,
             count(m) AS collaborations
    }
    RETURN gds.graph.project.remote(source, target, {
      sourceNodeLabels: labels(source),
      targetNodeLabels: labels(target),
      sourceNodeProperties: sourceNodeProperties,
      targetNodeProperties: targetNodeProperties,
      relationshipType: 'COLLABORATED',
      relationshipProperties: {collaborations: collaborations}
    })
    """
)

print(f"Projected graph: {G.name()}")
print(f"  Nodes: {G.node_count():,}")
print(f"  Relationships: {G.relationship_count():,}")

`G` is a `Graph` handle pointing at the projection inside the session. From here on, every method works the same way as local GDS.

## Step 6 — Stream mode

**`stream`** returns results as a DataFrame to Python. Use it when you want algorithm output for further analysis or visualisation in Python.

In [ ]:
# Stream degree centrality — the top 10 most-connected actors
df = gds.degree.stream(G)
display(df.nlargest(10, "score"))

The DataFrame has `nodeId` and `score` columns. `nodeId` is the session-internal ID, not the AuraDB node ID — we'll resolve back to actor names in a moment.

## Step 7 — Mutate mode

**`mutate`** stores the result as a property on the projection inside the session. The result isn't written to AuraDB — it lives only in the session's memory. Useful when one algorithm's output is another's input.

In [ ]:
# Mutate: store degree centrality on the projection
result = gds.degree.mutate(
    G,
    mutateProperty="rank"
)
print(f"Nodes processed: {result['nodePropertiesWritten']}")
print(f"Actor properties now: {G.node_properties('Actor')}")

The projection now has a `rank` property on every Actor. Nothing has been written to AuraDB yet.

## Step 8 — Write mode

**`write`** persists results to AuraDB. Use it when you want the scores available for future queries or want to visualise them in the Browser.

In [ ]:
# Write degree centrality back to AuraDB
result = gds.degree.write(
    G,
    writeProperty="degree"
)
print(f"Wrote degree centrality to {result['nodePropertiesWritten']:,} nodes")

## Step 9 — Verify the write

Query AuraDB directly to confirm the property landed.

In [ ]:
df = gds.run_cypher("""
    MATCH (a:Actor)
    WHERE a.degree IS NOT NULL
    RETURN a.name AS actor, a.degree AS degree
    ORDER BY a.degree DESC
    LIMIT 10
""")
display(df)

Top-10 actors by number of unique collaborators in the dataset.

## Step 10 — Clean up

End the session explicitly. The meter stops the moment the session terminates.

In [ ]:
sessions.delete(session_name="attached-demo")

You're done. The session is gone, and the `degree` property is persisted in AuraDB for any future query to use.

Move on to `3_standalone.ipynb` to do the same kind of workflow when your data isn't in Neo4j yet.